# SHAP Feature Importance Analysis

## Purpose
Compute and visualize global feature importance using SHAP (SHapley Additive exPlanations)
for the DockF+MDF model. Produces Figure 4 A/B and Figure 5 in the manuscript.

## Method
SHAP values are computed from out-of-fold (OOF) predictions generated during LOPOCV.
Importance is quantified as the mean absolute SHAP value across all OOF samples and
averaged over 20 independent random seeds.

## Input
- LOPOCV materials (model.txt, X_te.parquet, features.csv) from each seed/fold:
  `/path/to/output/seed_{1-20}/lgb_ds_md/{label_2p5,label_5p0}_lopocv/materials/fold*`

## Output
- Per-seed SHAP summary CSV: `seed_*/lgb_ds_md/*/feature_importance_shap.csv`
- Aggregated CSV: `imp_shap/shap_seed_summary_ds_md_{target}.csv`
- Figure 4 A/B: `imp_shap/shap_top10_ds_md_combined.{png,pdf,svg}`
- Figure 5: `visual/shap_beeswarm_ds_md_{target}.{png,pdf,svg}`

## Run Order
1. Run LOPOCV notebook (05) to generate materials
2. Run Cell 1 (SHAP recomputation) -- writes feature_importance_shap.csv per seed/fold
3. Run Cell 2 (SHAP bar plot) -- produces Figure 4 A/B
4. Run Cell 3 (SHAP beeswarm) -- produces Figure 5

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# ============================================================
# Cell 1: Recompute and save SHAP values from LOPOCV materials
# ============================================================
# Reads LightGBM model and OOF test data (X_te.parquet) saved
# during LOPOCV and computes mean|SHAP| per fold, then averages
# across folds to produce feature_importance_shap.csv per seed.
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import lightgbm as lgb

# ── Configuration ──────────────────────────────────────────────────────────
# Update OUT_DIR to match your local directory structure before running.
OUT_DIR   = Path("/path/to/analysis/output")  # root output directory
CONDITION = "ds_md"                            # model configuration
TARGETS   = ["label_2p5", "label_5p0"]        # RMSD thresholds (2.5 Å and 5.0 Å)
MATERIALS_SUBDIR = "materials"
SEEDS = None  # None → auto-detect from directory names
# ── End Configuration ──────────────────────────────────────────────────────

def detect_seeds(base: Path) -> list:
    seeds = []
    for p in base.glob("seed_*"):
        if p.is_dir():
            try:
                seeds.append(int(p.name.split("_")[1]))
            except Exception:
                pass
    return sorted(set(seeds))

def _read_features(fpath: Path):
    if not fpath.exists():
        return None
    return pd.read_csv(fpath).iloc[:, 0].astype(str).tolist()

def _fold_dirs(seed_dir: Path, condition: str, target: str) -> list:
    base = seed_dir / f"lgb_{condition}" / f"{target}_lopocv" / MATERIALS_SUBDIR
    if not base.exists():
        return []
    def _key(p):
        m = re.search(r"fold(\d+)_", p.name)
        return int(m.group(1)) if m else 10**9
    return sorted(
        [p for p in base.iterdir()
         if p.is_dir() and re.match(r"^fold\d+_[A-Za-z0-9]+$", p.name)],
        key=_key
    )

def rebuild_shap(seed: int, target: str):
    seed_dir = OUT_DIR / f"seed_{seed}"
    target_root = seed_dir / f"lgb_{CONDITION}" / f"{target}_lopocv"
    if not target_root.exists():
        return False

    shap_dir = target_root / "shap"
    shap_dir.mkdir(parents=True, exist_ok=True)

    fold_dirs = _fold_dirs(seed_dir, CONDITION, target)
    if not fold_dirs:
        return False

    per_fold = []
    for fdir in fold_dirs:
        model_path = fdir / "model.txt"
        x_te_path  = fdir / "X_te.parquet"
        feats_path = fdir / "features.csv"
        if not model_path.exists() or not x_te_path.exists():
            continue

        bst  = lgb.Booster(model_file=str(model_path))
        X_te = pd.read_parquet(x_te_path)
        feat_names = _read_features(feats_path)
        if feat_names:
            cols = [c for c in feat_names if c in X_te.columns]
            X_te = X_te[cols]
        else:
            cols = X_te.columns.tolist()

        # pred_contrib=True returns (n_samples, n_features+1); last col is bias
        shap_mat = bst.predict(X_te, pred_contrib=True)
        shap_vals = np.asarray(shap_mat)[:, :-1]
        mean_abs = np.abs(shap_vals).mean(axis=0)

        df_fold = pd.DataFrame({"feature": cols, "mean_abs_shap": mean_abs})
        df_fold.to_csv(shap_dir / f"mean_abs_shap_{fdir.name}.csv", index=False)

        s = df_fold.set_index("feature")["mean_abs_shap"]
        s.name = fdir.name
        per_fold.append(s)

    if not per_fold:
        return False

    df_cat = pd.concat(per_fold, axis=1).fillna(0.0)
    df_out = (
        df_cat.mean(axis=1)
              .rename("mean_abs_shap")
              .reset_index()
              .rename(columns={"index": "feature"})
              .sort_values("mean_abs_shap", ascending=False)
    )
    out_sum = target_root / "feature_importance_shap.csv"
    df_out.to_csv(out_sum, index=False)
    print(f"[OK] seed={seed}, target={target}: {out_sum.name}")
    return True

# Run
seeds = SEEDS or detect_seeds(OUT_DIR)
print(f"Seeds detected: {seeds}")
for s in seeds:
    for tgt in TARGETS:
        rebuild_shap(s, tgt)
print("Done — SHAP CSVs written.")

In [ ]:
# ============================================================
# Cell 2: SHAP summary bar plot (Figure 4 A/B)
# ============================================================
# Aggregates per-seed feature_importance_shap.csv files and
# plots mean |SHAP| ± SD for the top-10 features.
# Produces individual plots and a combined 2-panel figure.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────
# OUT_DIR, CONDITION, and TARGETS are defined in Cell 1.
SEEDS     = range(1, 21)   # seeds 1–20
TOPN_BAR  = 10
PLOT_DIR  = OUT_DIR / "imp_shap"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
# ── End Configuration ──────────────────────────────────────────────────────

# Font settings
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "mathtext.default": "regular",
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
})

PALETTE = {"DockF": "#ff7f0e", "LigF": "#2ca02c", "MDF": "#1f77b4"}

# Feature label dictionary (paper notation)
LABEL_MAP = {
    "cnn_affinity": "DockF: CNN affinity",
    "cnn_pose_score": "DockF: CNN pose score",
    "vina_affinity": "DockF: Vina affinity",
    "md_contact_per_atom_mean": "MDF: contacts/atom (mean)",
    "md_contact_per_atom_slope": "MDF: contacts/atom (slope)",
    "md_contact_per_atom_std": "MDF: contacts/atom (SD)",
    "md_contact_std": "MDF: contacts (SD)",
    "md_distance_mean": "MDF: ligand-binding-site distance (mean)",
    "md_distance_min": "MDF: ligand-binding-site distance (min)",
    "md_distance_p10": "MDF: ligand-binding-site distance (p10)",
    "md_distance_std": "MDF: ligand-binding-site distance (SD)",
    "md_eelec_mean": "MDF: electrostatic energy (mean)",
    "md_eelec_slope": "MDF: electrostatic energy (slope)",
    "md_eelec_std": "MDF: electrostatic energy (SD)",
    "md_evdw_slope": "MDF: van der Waals energy (slope)",
    "md_evdw_std": "MDF: van der Waals energy (SD)",
    "md_mmgbsa_delta_total": r"MDF: MM/GBSA $\Delta G_{\mathrm{bind}}$",
    "md_mmpbsa_delta_total": r"MDF: MM/PBSA $\Delta G_{\mathrm{bind}}$",
    "md_mmpbsa_std": "MDF: MM/PBSA (SD)",
    "md_rg_last": "MDF: radius of gyration (final)",
    "md_rg_slope": "MDF: radius of gyration (slope)",
    "md_rmsd_ca_last": "MDF: CA RMSD (final)",
    "md_rmsd_ca_mean": "MDF: CA RMSD (mean)",
    "md_rmsd_ca_min": "MDF: CA RMSD (min)",
    "md_rmsd_ca_slope": "MDF: CA RMSD (slope)",
    "md_rmsf_protein_mean": "MDF: protein RMSF (mean)",
    "md_contact_unique_residues": "MDF: unique contact residues",
    "md_prolif_any_contact_count_mean": "MDF: any contact count (mean)",
    "md_prolif_hbond_freq": "MDF: H-bond frequency",
    "md_prolif_ionic_count_mean": "MDF: ionic contact count (mean)",
    "md_prolif_interaction_entropy": "MDF: interaction entropy",
    "md_prolif_polar_ratio": "MDF: polar contact ratio",
    "md_prolif_aromatic_ratio": "MDF: aromatic contact ratio",
    "md_rmsd_lig_mean": "MDF: ligand RMSD (mean)",
    "md_rmsf_protein_max": "MDF: protein RMSF (max)",
    "md_rmsf_protein_std": "MDF: protein RMSF (SD)",
}

def feature_category(name):
    if name in {"vina_affinity", "cnn_pose_score", "cnn_affinity"}:
        return "DockF"
    if str(name).startswith("md_"):
        return "MDF"
    return "LigF"

def feature_color(name):
    return PALETTE.get(feature_category(name), "#7f7f7f")

def wrap_label(label, max_length=25):
    label = label.replace("DS: ", "DockF: ").replace("MD: ", "MDF: ")
    if len(label) <= max_length:
        return label
    if "$" in label and len(label.split("$")[0].strip()) <= 15:
        return label
    for i in range(max_length, max(0, max_length - 10), -1):
        if i < len(label) and label[i] in [" ", ":"]:
            return label[:i] + "\n" + label[i + 1:].lstrip()
    return label[:max_length] + "\n" + label[max_length:]

def format_label(name):
    return wrap_label(LABEL_MAP.get(name, name))

def load_seed_shap(seed, target):
    f = OUT_DIR / f"seed_{seed}" / f"lgb_{CONDITION}" / f"{target}_lopocv" / "feature_importance_shap.csv"
    if not f.exists():
        return None
    df = pd.read_csv(f)
    cols = list(df.columns)
    df = df.rename(columns={cols[0]: "feature", cols[1]: "mean_abs_shap"})
    df["seed"] = seed
    return df[["feature", "mean_abs_shap", "seed"]]

def aggregate_shap(target):
    rows = [load_seed_shap(s, target) for s in SEEDS]
    rows = [r for r in rows if r is not None]
    if not rows:
        raise RuntimeError(f"No SHAP CSVs found for {target}")
    cat = pd.concat(rows, ignore_index=True)
    g = cat.groupby("feature")["mean_abs_shap"]
    df_sum = pd.DataFrame({"feature": g.mean().index, "mean": g.mean().values,
                            "sd": g.std(ddof=1).values, "n": g.count().values})
    df_sum["category"] = df_sum["feature"].map(feature_category)
    out_csv = PLOT_DIR / f"shap_seed_summary_{CONDITION}_{target}.csv"
    df_sum.sort_values("mean", ascending=False).to_csv(out_csv, index=False)
    print(f"[OK] Saved: {out_csv.name}")
    return df_sum

# Aggregate and compute common x-axis limits
results = {}
for tgt in TARGETS:
    results[tgt] = aggregate_shap(tgt)

xmax = max(
    (results[t].sort_values("mean", ascending=False).head(TOPN_BAR)["mean"]
     + results[t].sort_values("mean", ascending=False).head(TOPN_BAR)["sd"].fillna(0)).max()
    for t in TARGETS
) * 1.08
xlim = (0.0, xmax)
print(f"[INFO] Common x-axis limit: {xlim}")

# Combined 2-panel plot (Figure 4 A/B)
fig, axes = plt.subplots(1, 2, figsize=(18, 8.5))
handles = [plt.Rectangle((0, 0), 1, 1, color=PALETTE[k]) for k in ["DockF", "MDF"]]

for idx, tgt in enumerate(TARGETS):
    ax = axes[idx]
    top = results[tgt].sort_values("mean", ascending=False).head(TOPN_BAR).iloc[::-1]
    ax.barh(range(len(top)), top["mean"], xerr=top["sd"].fillna(0.0),
            capsize=4, color=[feature_color(f) for f in top["feature"]],
            edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([format_label(f) for f in top["feature"]], fontsize=16)
    ax.set_xlabel("Mean |SHAP| ± SD", fontsize=16)
    ax.tick_params(axis="x", labelsize=14)
    rmsd_label = "2.5 Å" if tgt == "label_2p5" else "5.0 Å"
    ax.set_title(f"RMSD {rmsd_label}", fontsize=17, pad=12)
    ax.grid(axis="x", alpha=0.3)
    ax.set_xlim(xlim)

fig.legend(handles, ["DockF", "MDF"], loc="lower center",
           bbox_to_anchor=(0.5, 0.03), ncol=2, frameon=True, fontsize=17)
plt.tight_layout(pad=2.5, rect=[0, 0.07, 1, 1])

base = PLOT_DIR / f"shap_top{TOPN_BAR}_{CONDITION}_combined"
fig.savefig(str(base) + ".png", dpi=300, bbox_inches="tight")
fig.savefig(str(base) + ".pdf", bbox_inches="tight")
fig.savefig(str(base) + ".svg", bbox_inches="tight")
plt.show()
print(f"[OK] Saved Figure 4 A/B: {base.name}.*")

In [ ]:
# ============================================================
# Cell 3: SHAP beeswarm plot (Figure 5)
# ============================================================
# Loads SHAP values from LOPOCV materials (all seeds and folds)
# and plots a beeswarm showing the distribution of SHAP values
# for the top-20 features, colored by feature value (low=blue,
# high=red). Produces the combined 2-panel figure.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import lightgbm as lgb
import textwrap
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────
# OUT_DIR, CONDITION, and TARGETS are defined in Cell 1.
TOPN_SW   = 20
FIGSIZE   = (9.6, 7.0)
RNG       = np.random.default_rng(0)  # for reproducible jitter
# ── End Configuration ──────────────────────────────────────────────────────

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "mathtext.fontset": "dejavusans",
    "mathtext.default": "regular",
})

LABEL_MAP = {
    "cnn_affinity": "DockF: CNN affinity",
    "cnn_pose_score": "DockF: CNN pose score",
    "vina_affinity": "DockF: Vina affinity",
    "md_contact_per_atom_mean": "MDF: contacts/atom (mean)",
    "md_distance_p10": "MDF: ligand-binding-site distance (p10)",
    "md_distance_mean": "MDF: ligand-binding-site distance (mean)",
    "md_distance_min": "MDF: ligand-binding-site distance (min)",
    "md_eelec_mean": "MDF: electrostatic energy (mean)",
    "md_eelec_std": "MDF: electrostatic energy (SD)",
    "md_mmgbsa_delta_total": r"MDF: MM/GBSA $\Delta G_{\mathrm{bind}}$",
    "md_mmpbsa_delta_total": r"MDF: MM/PBSA $\Delta G_{\mathrm{bind}}$",
    "md_contact_std": "MDF: contacts (SD)",
    "md_rmsd_ca_mean": "MDF: CA RMSD (mean)",
    "md_contact_per_atom_slope": "MDF: contacts/atom (slope)",
    "md_eelec_slope": "MDF: electrostatic energy (slope)",
    "md_evdw_slope": "MDF: van der Waals energy (slope)",
    "md_evdw_std": "MDF: van der Waals energy (SD)",
    "md_rg_last": "MDF: radius of gyration (final)",
    "md_rmsd_ca_last": "MDF: CA RMSD (final)",
    "md_prolif_interaction_entropy": "MDF: interaction entropy",
}

def wrap2(s, width=18):
    s = LABEL_MAP.get(s, s)
    s = s.replace("DS: ", "DockF: ").replace("MD: ", "MDF: ")
    return textwrap.fill(s, width=width, break_long_words=False, break_on_hyphens=False)

def _seed_dirs():
    for p in sorted(OUT_DIR.glob("seed_*")):
        try:
            yield int(p.name.split("_")[1]), p
        except Exception:
            continue

def _fold_dirs(seed_dir, target):
    base = seed_dir / f"lgb_{CONDITION}" / f"{target}_lopocv" / "materials"
    if base.exists():
        for q in sorted(base.glob("fold*")):
            if q.is_dir():
                yield q

def _load_shap_fold(fold_dir, want_feats):
    model_path = fold_dir / "model.txt"
    x_path     = fold_dir / "X_te.parquet"
    feat_path  = fold_dir / "features.csv"
    if not (model_path.exists() and x_path.exists() and feat_path.exists()):
        return None
    feat_order = pd.read_csv(feat_path).iloc[:, 0].astype(str).tolist()
    X = pd.read_parquet(x_path)
    if any(c not in X.columns for c in feat_order):
        return None
    X_full = X[feat_order]
    bst = lgb.Booster(model_file=str(model_path))
    shap_raw = np.asarray(bst.predict(X_full.values, pred_contrib=True))[:, :-1]
    df_shap = pd.DataFrame(shap_raw, columns=feat_order, index=X_full.index)
    use_cols = [c for c in want_feats if c in df_shap.columns]
    if not use_cols:
        return None
    return df_shap[use_cols].copy(), X_full[use_cols].copy()

def plot_beeswarm(target, top_feats):
    shap_chunks, val_chunks = [], []
    for _, sdir in _seed_dirs():
        for fdir in _fold_dirs(sdir, target):
            got = _load_shap_fold(fdir, top_feats)
            if got is not None:
                shap_chunks.append(got[0])
                val_chunks.append(got[1])

    if not shap_chunks:
        raise FileNotFoundError(f"Could not collect SHAP from materials for {target}.")

    SH = pd.concat(shap_chunks, axis=0, ignore_index=True)
    XV = pd.concat(val_chunks,  axis=0, ignore_index=True)
    feats = [f for f in top_feats if f in SH.columns][::-1]
    pos = {f: i for i, f in enumerate(feats)}

    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=300)
    fig.subplots_adjust(left=0.32, right=0.96, top=0.95, bottom=0.12)

    for f in feats:
        s = SH[f].to_numpy()
        v = XV[f].to_numpy()
        s_lo, s_hi = np.nanpercentile(s, [1, 99])
        v_lo, v_hi = np.nanpercentile(v, [1, 99])
        s_clip = np.clip(s, s_lo, s_hi)
        v_norm = (np.clip(v, v_lo, v_hi) - v_lo) / (v_hi - v_lo + 1e-12)
        jitter = RNG.uniform(-0.25, 0.25, size=s_clip.shape)
        ax.scatter(s_clip, pos[f] + jitter, s=10, c=v_norm, cmap="coolwarm",
                   alpha=0.7, edgecolors="none", rasterized=True)

    ax.set_yticks(range(len(feats)), [wrap2(f) for f in feats], fontsize=10)
    ax.set_ylim(-0.5, len(feats) - 0.5)
    for t in ax.get_yticklabels():
        t.set_horizontalalignment("right")
        t.set_multialignment("right")
        t.set_linespacing(1.1)
    ax.spines[["top", "right"]].set_visible(False)
    ax.axvline(0.0, color="0.6", lw=1.2, ls="--")
    ax.set_xlabel("SHAP value", fontsize=12)
    rmsd_label = "2.5 Å" if target == "label_2p5" else "5.0 Å"
    ax.set_title(f"Feature importance (SHAP beeswarm)\nDockF+MDF (RMSD {rmsd_label})",
                 pad=12, fontsize=12)

    cbar = plt.colorbar(ax.collections[0], ax=ax, fraction=0.06, pad=0.02, aspect=35, shrink=0.75)
    cbar.set_label("Feature value (low → high)", fontsize=11)

    visual_dir = OUT_DIR / "visual"
    visual_dir.mkdir(parents=True, exist_ok=True)
    base = visual_dir / f"shap_beeswarm_{CONDITION}_{target}"
    fig.savefig(str(base) + ".png", dpi=300, bbox_inches="tight")
    fig.savefig(str(base) + ".pdf", bbox_inches="tight")
    fig.savefig(str(base) + ".svg", bbox_inches="tight")
    print(f"[OK] Saved Figure 5 ({target}): {base.name}.*")
    plt.show()

# Run beeswarm for both RMSD thresholds
for TARGET in TARGETS:
    rank_csv = OUT_DIR / "imp_shap" / f"shap_seed_summary_{CONDITION}_{TARGET}.csv"
    if not rank_csv.exists():
        print(f"[WARN] Run Cell 2 first to generate: {rank_csv.name}")
        continue
    top_feats = pd.read_csv(rank_csv)["feature"].head(TOPN_SW).tolist()
    print(f"\n[INFO] Beeswarm for {TARGET}...")
    plot_beeswarm(TARGET, top_feats)